In [14]:
%run manage_tables

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 45, Finished, Available, Finished, True)

In [15]:
%run transformation

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 52, Finished, Available, Finished, True)

In [16]:
from pyspark.sql.functions import *

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 53, Finished, Available, Finished, False)

In [17]:
def create_date_range_df(df, column):
    min_max = get_min_max_values(df, column)
    return spark.range(1).select(
        explode(
            sequence(
                lit(min_max["min_value"]),
                lit(min_max["max_value"]),
                lit(1).cast("interval day")
            )
        ).alias(column)
    )

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 54, Finished, Available, Finished, False)

In [18]:
def add_reverse_exchange_rate(df):
    return df.withColumn("usd_to_eur_multiplier", 1 / col("eur_to_usd_multiplier"))

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 55, Finished, Available, Finished, False)

In [19]:
def transform_exchange_rates(df):
    df = select_columns(df, ["TIME_PERIOD", "OBS_VALUE"])
    date_df = create_date_range_df(df, "TIME_PERIOD")
    df = join_dataframes(date_df, df, "TIME_PERIOD", how="left")
    df = fill_forward(df, "OBS_VALUE", "TIME_PERIOD")
    df = rename_columns(df, {"TIME_PERIOD": "date", "OBS_VALUE": "eur_to_usd_multiplier"})
    return add_reverse_exchange_rate(df)

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 56, Finished, Available, Finished, False)

In [20]:
def process():
    df = read_table("USD_EUR_FX_rates")
    df = transform_exchange_rates(df)
    save_table(df, "USD_EUR_FX_rates")
    return df

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 57, Finished, Available, Finished, False)

In [21]:
df = process()
display(df)

StatementMeta(, aa8a1dd3-b963-4cf8-a84f-109b3114e0a3, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c9c7b4c2-6541-448b-b1a0-3882a16f8092)